# 12. FP8 안정화 파이프라인 (U1/U2/U3)

## 구성 개요
- U1: BoN 기반 합성 데이터 생성 + LoRA SFT
- U2: On-policy GKD (선택)
- U3: Attention distillation (선택)
- 최종: FP8 static 양자화 + 로컬 반복 평가

In [ ]:
import gc
import json
import os
import random
import re
import shutil
import subprocess
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import Dataset, concatenate_datasets, load_dataset
from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import QuantizationModifier
from peft import LoraConfig, get_peft_model
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    default_data_collator,
    set_seed,
)

ROOT = Path("${PROJECT_ROOT}")
BASE_MODEL_DIR = ROOT / "open/base_model"
ANCHOR_MODEL_DIR = ROOT / "07_new_approaches/code/exp3_fp8_static_mixed512/model"
PRIORITY_MODEL_DIR = ROOT / "11_seed_ensemble_calibration_search/code/exp_se_exp_b1_fp8_static_m224_p224_a064_t512_k5_cap1024/model"
WORK_DIR = ROOT / "12_fp8_stability_pipeline/code"
EVAL_OUTPUT_DIR = ROOT / "12_fp8_stability_pipeline/eval_results"
EVAL_SCRIPT = ROOT / "eval/run_eval.py"
GLOBAL_SEED = 42

WORK_DIR.mkdir(parents=True, exist_ok=True)
EVAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

os.environ["TOKENIZERS_PARALLELISM"] = "false"
torch.set_float32_matmul_precision("high")
set_seed(GLOBAL_SEED)

print(f"기본 모델 경로   : {BASE_MODEL_DIR}")
print(f"앵커 모델 경로   : {ANCHOR_MODEL_DIR}")
print(f"1순위 모델 경로  : {PRIORITY_MODEL_DIR}")
print(f"작업 디렉토리    : {WORK_DIR}")
print(f"평가 출력 경로   : {EVAL_OUTPUT_DIR}")



In [ ]:
@dataclass
class ExperimentConfig:
    exp_id: str
    seed: int = 42

    # U1: 합성 데이터 + BoN
    n_prompts_total: int = 1200
    prompt_mix: Tuple[float, float, float] = (0.5, 0.3, 0.2)  # manta, gsm8k, pile
    bon_n: int = 6
    teacher_temp: float = 0.8
    teacher_top_p: float = 0.95
    synth_max_new_tokens: int = 192

    # 1단계 SFT (LoRA)
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    stage1_epochs: float = 0.6
    stage1_lr: float = 2e-4

    # U2/U3: 온폴리시 + KL 혼합 + (선택) 어텐션 증류
    use_onpolicy_gkd: bool = False
    onpolicy_n_prompts: int = 400
    stage2_epochs: float = 0.3
    stage2_lr: float = 1e-4
    temperature: float = 1.5
    ce_weight: float = 0.2
    fwd_kl_weight: float = 0.5
    rev_kl_weight: float = 0.3
    use_attention_loss: bool = False
    attn_layers: int = 2
    attn_weight: float = 0.05

    # 토크나이징 / 학습
    max_length: int = 1024
    train_bs: int = 1
    grad_accum: int = 16
    warmup_ratio: float = 0.03

    # FP8 정적 캘리브레이션
    calib_manta: int = 256
    calib_pile: int = 256
    calib_gsm: int = 0
    calib_max_length: int = 2048

    # 엄격 로컬 평가
    eval_n_runs: int = 7
    eval_tasks: str = "gsm8k,mmlu,arc_challenge"
    eval_limit: int = 512


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    set_seed(seed)


def exp_paths(cfg: ExperimentConfig) -> Dict[str, Path]:
    base = WORK_DIR / cfg.exp_id
    return {
        "base": base,
        "synthetic_jsonl": base / "synthetic_pairs.jsonl",
        "onpolicy_jsonl": base / "onpolicy_pairs.jsonl",
        "lora_dir": base / "lora",
        "lora_stage2_dir": base / "lora_stage2",
        "merged_fp16_dir": base / "merged_fp16",
        "model_dir": base / "model",
        "submit_zip": base / f"submit_{cfg.exp_id}.zip",
    }


def ensure_dirs(paths: Dict[str, Path]) -> None:
    for key, path in paths.items():
        if key.endswith("_zip") or path.suffix:
            path.parent.mkdir(parents=True, exist_ok=True)
        else:
            path.mkdir(parents=True, exist_ok=True)



In [ ]:
def pick_first_text(example: Dict, keys: List[str]) -> Optional[str]:
    for k in keys:
        v = example.get(k)
        if isinstance(v, str):
            t = v.strip()
            if t:
                return t
    for _, v in example.items():
        if isinstance(v, str):
            t = v.strip()
            if t:
                return t
    return None


def extract_prompt_from_manta(ex: Dict) -> Optional[str]:
    return pick_first_text(ex, ["instruction", "prompt", "question", "input", "text"])


def extract_prompt_from_gsm(ex: Dict) -> Optional[str]:
    return pick_first_text(ex, ["question", "prompt", "instruction"])


def extract_prompt_from_pile(ex: Dict) -> Optional[str]:
    txt = pick_first_text(ex, ["text"])
    if not txt:
        return None
    txt = re.sub(r"\s+", " ", txt).strip()[:700]
    return f"Summarize this passage in 3 sentences:\n\n{txt}"


def build_prompt_pool(cfg: ExperimentConfig) -> List[str]:
    seed_everything(cfg.seed)

    manta_n = int(cfg.n_prompts_total * cfg.prompt_mix[0])
    gsm_n = int(cfg.n_prompts_total * cfg.prompt_mix[1])
    pile_n = cfg.n_prompts_total - manta_n - gsm_n

    prompts: List[str] = []

    ds_manta = load_dataset("LGAI-EXAONE/MANTA-1M", split="train").shuffle(seed=cfg.seed).select(range(manta_n * 2))
    for ex in ds_manta:
        p = extract_prompt_from_manta(ex)
        if p:
            prompts.append(p)
        if len(prompts) >= manta_n:
            break

    ds_gsm = load_dataset("openai/gsm8k", "main", split="train").shuffle(seed=cfg.seed).select(range(gsm_n * 2))
    gsm_prompts = []
    for ex in ds_gsm:
        p = extract_prompt_from_gsm(ex)
        if p:
            gsm_prompts.append(p)
        if len(gsm_prompts) >= gsm_n:
            break
    prompts.extend(gsm_prompts)

    ds_pile = load_dataset("NeelNanda/pile-10k", split="train").shuffle(seed=cfg.seed).select(range(pile_n * 2))
    pile_prompts = []
    for ex in ds_pile:
        p = extract_prompt_from_pile(ex)
        if p:
            pile_prompts.append(p)
        if len(pile_prompts) >= pile_n:
            break
    prompts.extend(pile_prompts)

    random.Random(cfg.seed).shuffle(prompts)
    print(f"프롬프트 풀 크기: {len(prompts)} (manta={manta_n}, gsm8k={gsm_n}, pile={pile_n})")
    return prompts


def save_jsonl(rows: List[Dict], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


def load_jsonl(path: Path) -> List[Dict]:
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


In [ ]:
def load_teacher(base_model_dir: Path) -> Tuple[AutoModelForCausalLM, AutoTokenizer]:
    tok = AutoTokenizer.from_pretrained(base_model_dir, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        base_model_dir,
        torch_dtype=torch.float16,
        trust_remote_code=True,
        device_map="auto",
    )
    model.eval()
    return model, tok


def _format_bonus(prompt: str, answer: str) -> float:
    ans = answer.strip()
    bonus = 0.0
    if 1 <= len(ans) <= 220:
        bonus += 0.03
    if "As an AI" in ans or "I cannot" in ans:
        bonus -= 0.06
    if ("answer only" in prompt.lower() or "정답만" in prompt) and ("\n" not in ans):
        bonus += 0.02
    return bonus


@torch.no_grad()
def generate_best_of_n(
    model: AutoModelForCausalLM,
    tok: AutoTokenizer,
    prompt: str,
    n: int,
    temperature: float,
    top_p: float,
    max_new_tokens: int,
) -> Dict:
    messages = [{"role": "user", "content": prompt}]
    input_ids = tok.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(model.device)

    gen = model.generate(
        input_ids=input_ids,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        max_new_tokens=max_new_tokens,
        num_return_sequences=n,
        return_dict_in_generate=True,
        output_scores=True,
        pad_token_id=tok.eos_token_id,
    )

    seqs = gen.sequences
    gen_len = len(gen.scores)
    prompt_len = input_ids.shape[1]
    new_tok = seqs[:, prompt_len : prompt_len + gen_len]
    texts = tok.batch_decode(new_tok, skip_special_tokens=True)

    step_logprobs = []
    for t, step_scores in enumerate(gen.scores):
        lp = step_scores.float().log_softmax(dim=-1)
        chosen = new_tok[:, t].unsqueeze(-1)
        token_lp = lp.gather(1, chosen).squeeze(-1)
        step_logprobs.append(token_lp)
    step_logprobs = torch.stack(step_logprobs, dim=1)

    eos_id = tok.eos_token_id
    best_idx = 0
    best_score = -1e9
    for i, txt in enumerate(texts):
        row_toks = new_tok[i]
        eos_pos = (row_toks == eos_id).nonzero(as_tuple=False)
        valid_len = int(eos_pos[0].item() + 1) if eos_pos.numel() > 0 else row_toks.numel()
        valid_len = max(valid_len, 1)
        mean_lp = step_logprobs[i, :valid_len].mean().item()
        length_penalty = 0.01 * np.log1p(valid_len)
        score = mean_lp - length_penalty + _format_bonus(prompt, txt)
        if score > best_score:
            best_score = score
            best_idx = i

    return {
        "prompt": prompt,
        "answer": texts[best_idx].strip(),
        "bon_score": float(best_score),
        "n_candidates": n,
    }


def build_synthetic_pairs(cfg: ExperimentConfig, out_jsonl: Path, force_rebuild: bool = False) -> Dataset:
    if out_jsonl.exists() and not force_rebuild:
        rows = load_jsonl(out_jsonl)
        print(f"합성 데이터 재사용: {out_jsonl} ({len(rows)}개)")
        return Dataset.from_list(rows)

    prompts = build_prompt_pool(cfg)
    teacher, tok = load_teacher(BASE_MODEL_DIR)

    rows: List[Dict] = []
    t0 = time.time()
    for i, p in enumerate(prompts, start=1):
        row = generate_best_of_n(
            model=teacher,
            tok=tok,
            prompt=p,
            n=cfg.bon_n,
            temperature=cfg.teacher_temp,
            top_p=cfg.teacher_top_p,
            max_new_tokens=cfg.synth_max_new_tokens,
        )
        rows.append(row)
        if i % 50 == 0:
            dt = time.time() - t0
            print(f"[합성 생성] {i}/{len(prompts)} 완료 ({dt/60:.1f}분)")

    save_jsonl(rows, out_jsonl)

    del teacher
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return Dataset.from_list(rows)


In [ ]:
def tokenize_chat_pairs(ds: Dataset, tok: AutoTokenizer, max_length: int) -> Dataset:
    def _map_fn(ex: Dict) -> Dict:
        prompt = ex["prompt"].strip()
        answer = ex["answer"].strip()

        p_text = tok.apply_chat_template(
            [{"role": "user", "content": prompt}],
            tokenize=False,
            add_generation_prompt=True,
        )
        full_text = tok.apply_chat_template(
            [{"role": "user", "content": prompt}, {"role": "assistant", "content": answer}],
            tokenize=False,
            add_generation_prompt=False,
        )

        p_tok = tok(p_text, add_special_tokens=False, truncation=True, max_length=max_length)
        f_tok = tok(full_text, add_special_tokens=False, truncation=True, max_length=max_length)

        input_ids = f_tok["input_ids"]
        attention_mask = f_tok["attention_mask"]
        labels = input_ids.copy()

        p_len = min(len(p_tok["input_ids"]), len(labels))
        for i in range(p_len):
            labels[i] = -100

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
        }

    return ds.map(_map_fn, remove_columns=ds.column_names)


def load_lora_student(cfg: ExperimentConfig, model_dir: Path) -> Tuple[AutoModelForCausalLM, AutoTokenizer]:
    tok = AutoTokenizer.from_pretrained(model_dir, trust_remote_code=True)

    model = AutoModelForCausalLM.from_pretrained(
        model_dir,
        torch_dtype=torch.float16,
        trust_remote_code=True,
        device_map="auto",
    )
    model.config.use_cache = False

    lora_cfg = LoraConfig(
        r=cfg.lora_r,
        lora_alpha=cfg.lora_alpha,
        lora_dropout=cfg.lora_dropout,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
    )
    model = get_peft_model(model, lora_cfg)
    model.print_trainable_parameters()
    return model, tok


class GKDTrainer(Trainer):
    def __init__(
        self,
        teacher_model: AutoModelForCausalLM,
        temperature: float,
        ce_weight: float,
        fwd_kl_weight: float,
        rev_kl_weight: float,
        use_attention_loss: bool,
        attn_layers: int,
        attn_weight: float,
        **kwargs,
    ):
        super().__init__(**kwargs)
        self.teacher_model = teacher_model.eval()
        for p in self.teacher_model.parameters():
            p.requires_grad = False
        self.temperature = temperature
        self.ce_weight = ce_weight
        self.fwd_kl_weight = fwd_kl_weight
        self.rev_kl_weight = rev_kl_weight
        self.use_attention_loss = use_attention_loss
        self.attn_layers = attn_layers
        self.attn_weight = attn_weight

    def _masked_mean(self, x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        mask_f = mask.float()
        denom = mask_f.sum().clamp_min(1.0)
        return (x * mask_f).sum() / denom

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs["labels"]
        outputs_s = model(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            labels=labels,
            output_attentions=self.use_attention_loss,
        )
        ce_loss = outputs_s.loss

        with torch.no_grad():
            outputs_t = self.teacher_model(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                output_attentions=self.use_attention_loss,
            )

        t = self.temperature
        logits_s = outputs_s.logits[:, :-1, :]
        logits_t = outputs_t.logits[:, :-1, :]
        mask = labels[:, 1:] != -100

        logp_s = F.log_softmax(logits_s / t, dim=-1)
        logp_t = F.log_softmax(logits_t / t, dim=-1)
        p_s = logp_s.exp()
        p_t = logp_t.exp()

        fwd_tok = (p_t * (logp_t - logp_s)).sum(dim=-1)
        rev_tok = (p_s * (logp_s - logp_t)).sum(dim=-1)

        fwd_kl = self._masked_mean(fwd_tok, mask) * (t ** 2)
        rev_kl = self._masked_mean(rev_tok, mask) * (t ** 2)

        loss = (
            self.ce_weight * ce_loss
            + self.fwd_kl_weight * fwd_kl
            + self.rev_kl_weight * rev_kl
        )

        if self.use_attention_loss:
            n = min(self.attn_layers, len(outputs_s.attentions))
            attn_loss = torch.tensor(0.0, device=loss.device)
            for i in range(1, n + 1):
                s_attn = outputs_s.attentions[-i].mean(dim=1)
                t_attn = outputs_t.attentions[-i].mean(dim=1)
                attn_loss = attn_loss + F.mse_loss(s_attn, t_attn)
            attn_loss = attn_loss / max(n, 1)
            loss = loss + self.attn_weight * attn_loss

        return (loss, outputs_s) if return_outputs else loss


def build_training_args(out_dir: Path, lr: float, epochs: float, cfg: ExperimentConfig) -> TrainingArguments:
    return TrainingArguments(
        output_dir=str(out_dir),
        overwrite_output_dir=True,
        num_train_epochs=epochs,
        per_device_train_batch_size=cfg.train_bs,
        gradient_accumulation_steps=cfg.grad_accum,
        learning_rate=lr,
        warmup_ratio=cfg.warmup_ratio,
        lr_scheduler_type="cosine",
        logging_steps=10,
        save_strategy="no",
        eval_strategy="no",
        bf16=True,
        fp16=False,
        report_to=[],
        dataloader_num_workers=2,
        remove_unused_columns=False,
    )


@torch.no_grad()
def generate_onpolicy_pairs(
    student_model: AutoModelForCausalLM,
    tok: AutoTokenizer,
    prompts: List[str],
    max_new_tokens: int = 192,
) -> List[Dict]:
    rows = []
    student_model.eval()
    for i, p in enumerate(prompts, start=1):
        input_ids = tok.apply_chat_template(
            [{"role": "user", "content": p}],
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
        ).to(student_model.device)

        out = student_model.generate(
            input_ids=input_ids,
            do_sample=True,
            temperature=0.8,
            top_p=0.95,
            max_new_tokens=max_new_tokens,
            pad_token_id=tok.eos_token_id,
        )
        gen = out[0, input_ids.shape[1] :]
        ans = tok.decode(gen, skip_special_tokens=True).strip()
        rows.append({"prompt": p, "answer": ans, "source": "onpolicy"})
        if i % 50 == 0:
            print(f"[onpolicy] {i}/{len(prompts)}")
    return rows


In [ ]:
def build_calibration_dataset(cfg: ExperimentConfig, tok: AutoTokenizer) -> Dataset:
    parts = []

    if cfg.calib_manta > 0:
        manta = load_dataset("LGAI-EXAONE/MANTA-1M", split="train").shuffle(seed=cfg.seed).select(range(cfg.calib_manta))
        manta = manta.map(
            lambda x: {"text": pick_first_text(x, ["instruction", "prompt", "question", "input", "text"]) or ""},
            remove_columns=manta.column_names,
        )
        parts.append(manta)

    if cfg.calib_pile > 0:
        pile = load_dataset("NeelNanda/pile-10k", split="train").shuffle(seed=cfg.seed).select(range(cfg.calib_pile))
        pile = pile.map(
            lambda x: {"text": re.sub(r"\s+", " ", (x.get("text") or "").strip())[:800]},
            remove_columns=pile.column_names,
        )
        parts.append(pile)

    if cfg.calib_gsm > 0:
        gsm = load_dataset("openai/gsm8k", "main", split="train").shuffle(seed=cfg.seed).select(range(cfg.calib_gsm))
        gsm = gsm.map(lambda x: {"text": (x.get("question") or "").strip()}, remove_columns=gsm.column_names)
        parts.append(gsm)

    calib = concatenate_datasets(parts).shuffle(seed=cfg.seed)

    def _tok(ex):
        t = tok(ex["text"], truncation=True, max_length=cfg.calib_max_length, padding=False)
        return {"input_ids": t["input_ids"], "attention_mask": t["attention_mask"]}

    calib = calib.map(_tok, remove_columns=calib.column_names)
    return calib


def quantize_fp8_static(merged_fp16_dir: Path, output_model_dir: Path, cfg: ExperimentConfig) -> None:
    tok = AutoTokenizer.from_pretrained(merged_fp16_dir, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        merged_fp16_dir,
        torch_dtype=torch.float16,
        trust_remote_code=True,
        device_map="auto",
    )
    calib_ds = build_calibration_dataset(cfg, tok)
    n_calib = len(calib_ds)

    recipe = QuantizationModifier(
        targets="Linear",
        scheme="FP8",
        ignore=["lm_head"],
    )

    t0 = time.time()
    oneshot(
        model=model,
        dataset=calib_ds,
        recipe=recipe,
        num_calibration_samples=n_calib,
    )
    print(f"FP8 static 양자화 완료: {time.time() - t0:.1f}초, n_calib={n_calib}")

    output_model_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(output_model_dir, save_compressed=True)
    tok.save_pretrained(output_model_dir)

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def package_submit_zip(model_dir: Path, zip_path: Path) -> None:
    tmp_root = zip_path.parent / f"__tmp_submit_{int(time.time())}"
    tmp_model = tmp_root / "model"
    shutil.rmtree(tmp_root, ignore_errors=True)
    tmp_model.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(model_dir, tmp_model)
    base_name = str(zip_path.with_suffix(""))
    shutil.make_archive(base_name=base_name, format="zip", root_dir=tmp_root)
    shutil.rmtree(tmp_root, ignore_errors=True)
    print(f"제출 ZIP 생성 완료: {zip_path} ({zip_path.stat().st_size / (1024**2):.1f} MB)")


def run_strict_eval(
    model_paths: List[Path],
    baseline_model: Path = ANCHOR_MODEL_DIR,
    n_runs: int = 7,
    tasks: str = "gsm8k,mmlu,arc_challenge",
    limit: int = 512,
    output_dir: Path = EVAL_OUTPUT_DIR,
) -> pd.DataFrame:
    cmd = [
        "python",
        str(EVAL_SCRIPT),
        "--model_path",
        str(baseline_model),
        *[str(p) for p in model_paths],
        "--n_runs",
        str(n_runs),
        "--baseline_model_idx",
        "0",
        "--tasks",
        tasks,
        "--limit",
        str(limit),
        "--output_dir",
        str(output_dir),
    ]
    print("로컬 평가 명령:")
    print(" ".join(cmd))
    subprocess.run(cmd, check=True)

    cmp_files = sorted((ROOT / "eval").glob("comparison_*.json"))
    if not cmp_files:
        raise FileNotFoundError("comparison_*.json not found under ${PROJECT_ROOT}/eval")
    latest = cmp_files[-1]
    print(f"최신 comparison 파일: {latest}")

    rows = json.loads(latest.read_text(encoding="utf-8"))
    df = pd.DataFrame(rows)
    if "time_stats" in df.columns:
        df["time_stdev"] = df["time_stats"].apply(
            lambda x: float(x.get("stdev", 0.0)) if isinstance(x, dict) else np.nan
        )
        df["time_median"] = df["time_stats"].apply(
            lambda x: float(x.get("median", np.nan)) if isinstance(x, dict) else np.nan
        )
    else:
        df["time_stdev"] = np.nan
        df["time_median"] = np.nan

    if "score_proxy" in df.columns:
        # variance-penalized proxy for safer submission pick
        df["risk_adjusted_proxy"] = df["score_proxy"] - 0.01 * df["time_stdev"].fillna(0.0)
    else:
        df["risk_adjusted_proxy"] = np.nan

    view_cols = [
        "model_name",
        "overall_median",
        "score_proxy",
        "perf_norm_proxy",
        "speed_norm_proxy",
        "time_median",
        "time_stdev",
        "risk_adjusted_proxy",
        "model_path",
    ]
    view_cols = [c for c in view_cols if c in df.columns]
    return df.sort_values(by="risk_adjusted_proxy", ascending=False)[view_cols]


In [ ]:
def run_experiment(cfg: ExperimentConfig, force_rebuild_synth: bool = False) -> Dict:
    seed_everything(cfg.seed)
    paths = exp_paths(cfg)
    ensure_dirs(paths)
    print(json.dumps(asdict(cfg), ensure_ascii=False, indent=2))

    # 1) U1 합성 데이터 + BoN
    synth_ds = build_synthetic_pairs(cfg, paths["synthetic_jsonl"], force_rebuild=force_rebuild_synth)
    base_tok = AutoTokenizer.from_pretrained(BASE_MODEL_DIR, trust_remote_code=True)
    train_ds = tokenize_chat_pairs(synth_ds, base_tok, cfg.max_length)
    print(f"토큰화된 합성 학습 데이터 크기: {len(train_ds)}")

    # 2) 1단계 LoRA SFT
    student, tok = load_lora_student(cfg, BASE_MODEL_DIR)
    args1 = build_training_args(paths["lora_dir"], lr=cfg.stage1_lr, epochs=cfg.stage1_epochs, cfg=cfg)
    trainer1 = Trainer(
        model=student,
        args=args1,
        train_dataset=train_ds,
        data_collator=default_data_collator,
        tokenizer=tok,
    )
    trainer1.train()
    student.save_pretrained(paths["lora_dir"])
    tok.save_pretrained(paths["lora_dir"])

    # 3) (선택) U2/U3 온폴리시 + KL 혼합
    if cfg.use_onpolicy_gkd:
        prompt_rows = synth_ds.select(range(min(cfg.onpolicy_n_prompts, len(synth_ds))))
        prompts = [r["prompt"] for r in prompt_rows]
        onpolicy_rows = generate_onpolicy_pairs(student, tok, prompts, max_new_tokens=cfg.synth_max_new_tokens)
        save_jsonl(onpolicy_rows, paths["onpolicy_jsonl"])
        onpolicy_ds = Dataset.from_list(onpolicy_rows)
        onpolicy_tok_ds = tokenize_chat_pairs(onpolicy_ds, tok, cfg.max_length)
        print(f"토큰화된 온폴리시 데이터 크기: {len(onpolicy_tok_ds)}")

        teacher = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL_DIR,
            torch_dtype=torch.float16,
            trust_remote_code=True,
            device_map="auto",
        )
        teacher.eval()

        args2 = build_training_args(paths["lora_stage2_dir"], lr=cfg.stage2_lr, epochs=cfg.stage2_epochs, cfg=cfg)
        trainer2 = GKDTrainer(
            model=student,
            args=args2,
            train_dataset=onpolicy_tok_ds,
            data_collator=default_data_collator,
            tokenizer=tok,
            teacher_model=teacher,
            temperature=cfg.temperature,
            ce_weight=cfg.ce_weight,
            fwd_kl_weight=cfg.fwd_kl_weight,
            rev_kl_weight=cfg.rev_kl_weight,
            use_attention_loss=cfg.use_attention_loss,
            attn_layers=cfg.attn_layers,
            attn_weight=cfg.attn_weight,
        )
        trainer2.train()
        student.save_pretrained(paths["lora_stage2_dir"])

        del teacher
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # 4) LoRA 병합 -> FP16
    merged = student.merge_and_unload()
    paths["merged_fp16_dir"].mkdir(parents=True, exist_ok=True)
    merged.save_pretrained(paths["merged_fp16_dir"])
    tok.save_pretrained(paths["merged_fp16_dir"])
    del student, merged
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # 5) FP8 정적 양자화
    quantize_fp8_static(paths["merged_fp16_dir"], paths["model_dir"], cfg)

    # 6) submit.zip 패키징
    package_submit_zip(paths["model_dir"], paths["submit_zip"])

    return {
        "exp_id": cfg.exp_id,
        "model_dir": str(paths["model_dir"]),
        "submit_zip": str(paths["submit_zip"]),
        "synthetic_jsonl": str(paths["synthetic_jsonl"]),
    }



In [ ]:
EXPERIMENTS = [
    ExperimentConfig(
        exp_id="exp_u1_bon6_static_anchor",
        seed=42,
        use_onpolicy_gkd=False,
        use_attention_loss=False,
        bon_n=6,
        n_prompts_total=1200,
        calib_manta=256,
        calib_pile=256,
        calib_gsm=0,
        eval_n_runs=7,
    ),
    ExperimentConfig(
        exp_id="exp_u1_u2_klmix_static",
        seed=42,
        use_onpolicy_gkd=True,
        use_attention_loss=False,
        bon_n=6,
        n_prompts_total=1200,
        onpolicy_n_prompts=400,
        ce_weight=0.2,
        fwd_kl_weight=0.5,
        rev_kl_weight=0.3,
        calib_manta=256,
        calib_pile=224,
        calib_gsm=32,
        eval_n_runs=7,
    ),
    ExperimentConfig(
        exp_id="exp_u1_u2_u3_attn2_static",
        seed=42,
        use_onpolicy_gkd=True,
        use_attention_loss=True,
        attn_layers=2,
        attn_weight=0.05,
        bon_n=8,
        n_prompts_total=1400,
        onpolicy_n_prompts=500,
        ce_weight=0.15,
        fwd_kl_weight=0.5,
        rev_kl_weight=0.35,
        calib_manta=224,
        calib_pile=224,
        calib_gsm=64,
        eval_n_runs=9,
    ),
]

pd.DataFrame([asdict(x) for x in EXPERIMENTS]).T


In [ ]:
# 실행 스위치
RUN_BUILD = True          # True면 모델 생성 실행
RUN_LOCAL_EVAL = False     # True면 노트북에서 바로 로컬 평가 실행
FORCE_REBUILD_SYNTH = False

print("[실행 설정]")
print(f"- RUN_BUILD={RUN_BUILD}")
print(f"- RUN_LOCAL_EVAL={RUN_LOCAL_EVAL}")
print(f"- FORCE_REBUILD_SYNTH={FORCE_REBUILD_SYNTH}")

# 1) 모델 생성
built = []
if RUN_BUILD:
    print("\n[모델 생성 시작]")
    for cfg in EXPERIMENTS:
        built.append(run_experiment(cfg, force_rebuild_synth=FORCE_REBUILD_SYNTH))

    print("\n[모델 생성 완료]")
    for x in built:
        print(f"- {x['exp_id']} | model={x['model_dir']} | zip={x['submit_zip']}")
else:
    print("\nRUN_BUILD=False: 모델 생성을 건너뜁니다.")

# 2) 후보 모델 경로 수집 (새로 생성 + 기존 생성 모두 포함)
candidate_model_dirs = []
for cfg in EXPERIMENTS:
    p_model = WORK_DIR / cfg.exp_id / "model"
    if (p_model / "config.json").exists():
        candidate_model_dirs.append(p_model)

if built:
    candidate_model_dirs = [Path(x["model_dir"]) for x in built]

# 중복 제거
seen = set()
candidate_model_dirs = [p for p in candidate_model_dirs if not (str(p) in seen or seen.add(str(p)))]

print("\n[신규 후보 모델 목록]")
if candidate_model_dirs:
    for p in candidate_model_dirs:
        print(f"- {p}")
else:
    print("- (없음) 먼저 RUN_BUILD=True로 모델을 생성하세요.")

# 3) 고정 비교 모델(mixed512 + 현재 1순위) 구성
fixed_compare_models = []
if PRIORITY_MODEL_DIR.exists() and PRIORITY_MODEL_DIR != ANCHOR_MODEL_DIR:
    fixed_compare_models.append(PRIORITY_MODEL_DIR)

print("\n[고정 비교 모델]")
print(f"- 기준(앵커): {ANCHOR_MODEL_DIR}")
if fixed_compare_models:
    for p in fixed_compare_models:
        print(f"- 비교: {p}")
else:
    print("- 비교: (없음)")

# 4) 최종 평가 대상 구성: 고정 비교 모델 + 신규 후보
all_eval_models = fixed_compare_models + candidate_model_dirs
seen = set()
all_eval_models = [p for p in all_eval_models if not (str(p) in seen or seen.add(str(p)))]

print("\n[최종 로컬 평가 대상]")
if all_eval_models:
    for p in all_eval_models:
        print(f"- {p}")
else:
    print("- (없음)")

# 5) CLI 출력
nb_path = WORK_DIR / "Experiments_FP8_StabilityPipeline.ipynb"
model_build_cli = (
    f"jupyter nbconvert --to notebook --execute {nb_path} "
    f"--output Experiments_FP8_StabilityPipeline.executed.ipynb "
    f"--ExecutePreprocessor.timeout=-1"
)

print("\n[모델 생성 CLI (노트북 전체 자동 실행)]")
print(model_build_cli)

if all_eval_models:
    eval_cmd = (
        f"python {EVAL_SCRIPT} "
        f"--model_path {ANCHOR_MODEL_DIR} "
        f"{' '.join(str(p) for p in all_eval_models)} "
        f"--n_runs {max(cfg.eval_n_runs for cfg in EXPERIMENTS)} "
        f"--baseline_model_idx 0 "
        f"--tasks {EXPERIMENTS[0].eval_tasks} "
        f"--limit {EXPERIMENTS[0].eval_limit} "
        f"--output_dir {EVAL_OUTPUT_DIR}"
    )

    print("\n[로컬 평가 CLI]")
    print(eval_cmd)

    if RUN_LOCAL_EVAL:
        print("\n[노트북 내 로컬 평가 실행]")
        df_eval = run_strict_eval(
            model_paths=all_eval_models,
            baseline_model=ANCHOR_MODEL_DIR,
            n_runs=max(cfg.eval_n_runs for cfg in EXPERIMENTS),
            tasks=EXPERIMENTS[0].eval_tasks,
            limit=EXPERIMENTS[0].eval_limit,
            output_dir=EVAL_OUTPUT_DIR,
        )
        display(df_eval)
    else:
        print("RUN_LOCAL_EVAL=False: 평가 실행은 건너뛰고 CLI만 출력합니다.")
else:
    print("\n로컬 평가 대상 모델이 없어 CLI를 생성하지 않았습니다.")




## 권장 실행 순서
1. `RUN_BUILD=True`로 모델 생성
2. 출력된 `submit_*.zip` 및 `model` 경로 확인
3. `RUN_LOCAL_EVAL=True` 또는 출력된 평가 CLI를 터미널에서 실행
4. `12_fp8_stability_pipeline/eval_results`와 `${PROJECT_ROOT}/eval/comparison_*.json` 비교
5. 최종 제출 후보를 `mixed512`, `현재 1순위`, `신규 실험` 순으로 재정렬

